*****************************************************************************
# NC RISCC ECOSYSTEM TRANSFORMATIONS — DATA DOWNLOAD
*****************************************************************************

# *Download Ecosystem Transformation Data from CyVerse*

This notebook uses `gocmd` to pull ecosystem transformation datasets from the CyVerse Data Store into the local `data/` folder of this repository.

**Available datasets:**

| Key | Description | CyVerse Path |
|-----|-------------|---------------|
| `area_transformed` | Area transformed | `/iplant/home/shared/earthlab/NC_eco_transformations/biomass_transformations/Transformations` |
| `transformation_risk` | Transformation risk | `/iplant/home/shared/earthlab/NC_eco_transformations/biomass_transformations/transformation_risk` |
| `biomass_change` | Biomass change | `/iplant/home/shared/earthlab/NC_eco_transformations/biomass_transformations/biomass_change` |
| `total_biomass` | Total biomass | `/iplant/home/shared/earthlab/NC_eco_transformations/biomass_transformations/Total_Biomass_combined` |

**Usage:** Edit `DATASETS_TO_DOWNLOAD` in the configuration cell to select which datasets to pull, then run all cells.

---
## 1. Configuration

In [ ]:
import os

# ── CyVerse source paths ───────────────────────────────────────────────────────
CYVERSE_BASE = "i:/iplant/home/shared/earthlab/NC_eco_transformations/biomass_transformations"

DATASETS = {
    "area_transformed":    f"{CYVERSE_BASE}/Transformations",
    "transformation_risk": f"{CYVERSE_BASE}/transformation_risk",
    "biomass_change":      f"{CYVERSE_BASE}/biomass_change",
    "total_biomass":       f"{CYVERSE_BASE}/Total_Biomass_combined",
}

# ── Local destination ──────────────────────────────────────────────────────────
# Resolves to the `data/` folder at the repository root regardless of where
# the notebook is launched from.
REPO_ROOT  = os.path.abspath(os.path.join(os.getcwd(), ".."))
LOCAL_BASE = os.path.join(REPO_ROOT, "data", "eco_transformations")

# ── Select datasets to download ────────────────────────────────────────────────
# Set to a list of keys from DATASETS, or use "all" to download everything.
DATASETS_TO_DOWNLOAD = "all"   # e.g. ["area_transformed", "biomass_change"]

print(f"Repository root : {REPO_ROOT}")
print(f"Download target : {LOCAL_BASE}")

---
## 2. Resolve Dataset Selection

In [ ]:
if DATASETS_TO_DOWNLOAD == "all":
    selected = DATASETS
else:
    unknown = set(DATASETS_TO_DOWNLOAD) - set(DATASETS)
    if unknown:
        raise ValueError(f"Unknown dataset key(s): {unknown}. Valid keys: {list(DATASETS)}")
    selected = {k: DATASETS[k] for k in DATASETS_TO_DOWNLOAD}

print("Datasets queued for download:")
for name, path in selected.items():
    print(f"  {name:25s} -> {path}")

---
## 3. Create Local Directories

In [ ]:
for name in selected:
    local_dir = os.path.join(LOCAL_BASE, name)
    os.makedirs(local_dir, exist_ok=True)
    print(f"Ready: {local_dir}")

---
## 4. Download Data via `gocmd`

In [ ]:
for name, cyverse_path in selected.items():
    local_dir = os.path.join(LOCAL_BASE, name)
    print(f"\n{'='*60}")
    print(f"Downloading: {name}")
    print(f"  From : {cyverse_path}")
    print(f"  To   : {local_dir}")
    print(f"{'='*60}")
    !yes a | gocmd get {cyverse_path} {local_dir}
    print(f"Done: {name}")

---
## 5. Verify Downloads

In [ ]:
print("Download summary\n" + "-" * 40)
for name in selected:
    local_dir = os.path.join(LOCAL_BASE, name)
    files = []
    for root, _, fnames in os.walk(local_dir):
        for f in fnames:
            files.append(os.path.join(root, f))
    total_mb = sum(os.path.getsize(f) for f in files) / 1e6
    print(f"  {name:25s}: {len(files):>5d} file(s)   {total_mb:>8.2f} MB")